# SIH 2026: Crop Disease Detection (PlantVillage 15 Classes)
## EfficientNetB0 Transfer Learning & Fine-Tuning Pipeline

### Exact Model & Training Specifications:
- **Base Model**: EfficientNetB0 (Pretrained on ImageNet, include_top=False, input_shape=(224, 224, 3))
- **Custom Head**:
  - `GlobalAveragePooling2D()`
  - `Dense(128, activation='relu')`
  - `Dropout(0.3)`
  - `Dense(num_classes, activation='softmax')` where `num_classes=15`
- **Phase 1 (Feature Extraction)**:
  - Base model frozen
  - Adam optimizer, learning rate = `0.0001` (1e-4)
  - Loss: `categorical_crossentropy`, Metrics: `['accuracy']`
  - Epochs: `10`
- **Phase 2 (Fine-Tuning)**:
  - Unfreeze last 30% of EfficientNetB0 layers (first 70% remain frozen)
  - Lower learning rate = `0.00001` (1e-5)
  - Continue training for `15 more epochs` (Total: 25 epochs)
- **After Training**:
  - Evaluate on `test_data`, print test accuracy
  - Save as `'crop_disease_model.h5'`
  - Convert & export to `'crop_disease_model.tflite'` for offline edge inference

In [ ]:
# 1. Environment Check (Verify Colab T4 GPU)
!nvidia-smi
import tensorflow as tf
print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

### 2. Dataset Setup (PlantVillage 15 Classes)

In [ ]:
!pip install -q kaggle
import os
from google.colab import files

# Upload kaggle.json if not present
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Upload your kaggle.json file:")
    files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d emmarex/plantdisease -p ./dataset --unzip
print("[✓] Dataset ready!")

### 3. Load & Split Data (70% Train, 15% Val, 15% Test) with Augmentation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATASET_DIR = "dataset/plantvillage"

# Data Augmentation: Rotation +-20 deg, horizontal flip, zoom 0.1, contrast +-15%
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.055, fill_mode="nearest", name="rotation_pm20_deg"),
    layers.RandomFlip("horizontal", name="horizontal_flip"),
    layers.RandomZoom((-0.1, 0.1), fill_mode="nearest", name="zoom_0.1"),
    layers.RandomContrast(0.15, name="brightness_contrast_pm15_pct")
], name="data_augmentation")

# Load dataset and split
full_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    seed=42,
    shuffle=True
)
class_names = full_ds.class_names
num_classes = len(class_names)
total_batches = len(full_ds)

train_batches = int(total_batches * 0.70)
val_batches = int(total_batches * 0.15)

train_data = full_ds.take(train_batches)
remaining = full_ds.skip(train_batches)
val_data = remaining.take(val_batches)
test_data = remaining.skip(val_batches)

print(f"[+] Total Classes: {num_classes}")
print(f"[+] Train Batches: {train_batches} (70%) | Val Batches: {val_batches} (15%) | Test Batches: {len(test_data)} (15%)")

AUTOTUNE = tf.data.AUTOTUNE
train_data = train_data.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_data = val_data.cache().prefetch(buffer_size=AUTOTUNE)
test_data = test_data.cache().prefetch(buffer_size=AUTOTUNE)

### 4. Build Model: EfficientNetB0 + Exact Custom Classification Head

In [ ]:
inputs = layers.Input(shape=(224, 224, 3), name="input_leaf_image")
augmented = data_augmentation(inputs)

# Pretrained EfficientNetB0 Base (include_top=False, input_shape=(224,224,3))
base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_tensor=augmented
)
base_model.trainable = False  # FROZEN initially

# Exact Custom Head:
# - GlobalAveragePooling2D()
# - Dense(128, activation='relu')
# - Dropout(0.3)
# - Dense(num_classes, activation='softmax') where num_classes=15
x = layers.GlobalAveragePooling2D(name="global_avg_pool")(base_model.output)
x = layers.Dense(128, activation="relu", name="dense_128_relu")(x)
x = layers.Dropout(0.3, name="dropout_0.3")(x)
outputs = layers.Dense(num_classes, activation="softmax", name="disease_prediction")(x)

model = Model(inputs=inputs, outputs=outputs, name="CropDisease_EfficientNetB0")
model.summary()

### 5. Phase 1 (Feature Extraction): Base Frozen, LR = 0.0001, Epochs = 10

In [ ]:
phase1_optimizer = Adam(learning_rate=0.0001)
model.compile(
    optimizer=phase1_optimizer,
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_p1 = [
    EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ModelCheckpoint("best_checkpoint.h5", monitor="val_accuracy", save_best_only=True, verbose=1)
]

print("Starting Phase 1 (10 Epochs, Base Frozen)...")
history_phase1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks_p1
)
print(f"[✓] Phase 1 Final Val Accuracy: {history_phase1.history['val_accuracy'][-1]*100:.2f}%")

### 6. Phase 2 (Fine-Tuning): Unfreeze Last 30% Layers, LR = 0.00001, 15 More Epochs

In [ ]:
# Unfreeze last 30% of EfficientNetB0 layers
base_model.trainable = True
total_layers = len(base_model.layers)
unfreeze_count = int(total_layers * 0.30)
freeze_until = total_layers - unfreeze_count

for layer in base_model.layers[:freeze_until]:
    layer.trainable = False
for layer in base_model.layers[freeze_until:]:
    layer.trainable = True

print(f"Total base layers: {total_layers} | Frozen (First 70%): {freeze_until} | Trainable (Last 30%): {unfreeze_count}")

phase2_optimizer = Adam(learning_rate=0.00001)
model.compile(
    optimizer=phase2_optimizer,
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_p2 = [
    EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1),
    ModelCheckpoint("crop_disease_model.h5", monitor="val_accuracy", save_best_only=True, verbose=1)
]

initial_epoch = history_phase1.epoch[-1] + 1
total_epochs = initial_epoch + 15

print(f"Starting Phase 2 Fine-Tuning (Epochs {initial_epoch+1} to {total_epochs})...")
history_phase2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=total_epochs,
    initial_epoch=initial_epoch,
    callbacks=callbacks_p2
)
print(f"[✓] Phase 2 Final Val Accuracy: {history_phase2.history['val_accuracy'][-1]*100:.2f}%")

### 7. Evaluation on Test Data & Metrics Visualization

In [ ]:
# Evaluate on held-out test data
test_loss, test_accuracy = model.evaluate(test_data)
print(f"\n[✓] >>> Final Test Accuracy: {test_accuracy * 100:.2f}% <<<")
print(f"[✓] Final Test Loss: {test_loss:.4f}")

# Plot accuracy and loss curves
acc = history_phase1.history['accuracy'] + history_phase2.history['accuracy']
val_acc = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']
loss = history_phase1.history['loss'] + history_phase2.history['loss']
val_loss = history_phase1.history['val_loss'] + history_phase2.history['val_loss']

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Acc', color='#2D6A4F', linewidth=2)
plt.plot(val_acc, label='Validation Acc', color='#E76F51', linewidth=2)
plt.axhline(y=0.85, color='gray', linestyle='--', label='85% Benchmark')
plt.title('Training & Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss', color='#2D6A4F', linewidth=2)
plt.plot(val_loss, label='Validation Loss', color='#E76F51', linewidth=2)
plt.title('Loss Convergence Curves')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 8. Save Trained Model (.h5) & Export to TensorFlow Lite (.tflite)

In [ ]:
# 1. Save native Keras format
h5_filename = 'crop_disease_model.h5'
model.save(h5_filename)
print(f"[✓] Keras model saved as: '{h5_filename}' ({os.path.getsize(h5_filename)/(1024*1024):.2f} MB)")

# 2. Convert to TensorFlow Lite (.tflite) with dynamic range quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_filename = 'crop_disease_model.tflite'
with open(tflite_filename, 'wb') as f:
    f.write(tflite_model)

print(f"[✓] TFLite model exported as: '{tflite_filename}' ({os.path.getsize(tflite_filename)/(1024*1024):.2f} MB)")
print("[✓] Ready for offline mobile/web edge deployment!")


# 3. Save class names mapping to classes.json
import json
classes_filename = 'classes.json'
with open(classes_filename, 'w') as f:
    json.dump(class_names, f, indent=2)
print(f"[✓] Class labels exported as: '{classes_filename}' ({len(class_names)} classes)")

# 4. Download artifacts directly to local machine
files.download(h5_filename)
files.download(tflite_filename)
files.download(classes_filename)
